# Does a ridge penalty help HAR?

HAR-ridge fits HAR's slopes by least squares plus a penalty
$\alpha\,\lVert b \rVert^2$, which pulls them towards zero and the forecast
towards the mean. Shrinkage often helps when estimates are noisy. This notebook
asks, for every asset, whether a penalty chosen without looking at the
evaluation period beats plain HAR.

Methods are explained in `02_har`. Every number that depends on the data is
printed by the code cell it comes from.

In [1]:
from datetime import timedelta

import _display
import pandas as pd
from _display import by_asset, model_table, pair
from design import ASSETS, DESIGN, HAR, HAR_FEATURES, LAYOUT, LOG_RV, LOG_RV_MEANS

from rvol.data.preparation import build_forecast_data
from rvol.data.volatility import daily_dataset
from rvol.evaluation import forecast_losses, results_table
from rvol.models import ForecastModel, LinearRegression
from rvol.walkforward import smeared_forecasts, walk_forward

In [2]:
# Helpers used only by this notebook.
# Short names for the design's choices.
CANDIDATE, LOSS_LABEL = DESIGN.candidate, DESIGN.loss_name


LOWER, WINS = f"{CANDIDATE} lower by", f"{CANDIDATE} wins"


def ridge_har(penalty):
    return ForecastModel("HAR-ridge", LinearRegression(ridge=penalty), HAR_FEATURES)


def calibration(prepared):
    # Mean loss of each penalty on the calibration window: targets before the evaluation start.
    scores = {}
    for penalty in RIDGE_GRID:
        log_forecasts = walk_forward(
            prepared,
            ridge_har(penalty),
            min_train_size=DESIGN.min_train_size,
            start=DESIGN.forecast_start,
            end=DESIGN.evaluation_start - timedelta(days=1),
        )
        losses = forecast_losses(smeared_forecasts(log_forecasts), DESIGN.loss)
        scores[penalty] = float(losses["loss"].mean())
    return pd.Series(scores, name=f"calibration {LOSS_LABEL}").rename_axis("ridge penalty")


def hindsight(run):
    # Every penalty scored on the evaluation period: shows, chooses nothing.
    scores = {}
    for penalty in RIDGE_GRID:
        ridge = DESIGN.forecast(run.data, [ridge_har(penalty)], series=run.series)
        har = run.forecasts.query("model == @CANDIDATE")
        summary = DESIGN.evaluate(pd.concat([har, ridge])).summary.set_index("model")
        scores[penalty] = summary.loc["HAR-ridge", "mean_loss"]
    return pd.Series(scores, name=f"evaluation {LOSS_LABEL}, with hindsight").rename_axis(
        "ridge penalty"
    )


def verdict(asset):
    row = results_table(RUNS[asset].evaluation).loc["HAR-ridge"]
    penalty = CALIBRATION[asset].idxmin()
    edge = " (the edge of the grid)" if penalty in (min(RIDGE_GRID), max(RIDGE_GRID)) else ""
    outcome = "loses to" if row[LOWER] > 0 else "beats"
    return (
        f"{pair(asset)}: chosen penalty {penalty:g}{edge}; HAR-ridge {outcome} {CANDIDATE}: "
        f"{CANDIDATE}'s mean {LOSS_LABEL} is {abs(row[LOWER]):.1%} "
        f"{'below' if row[LOWER] > 0 else 'above'} HAR-ridge's (DM p-value {row['DM p-value']:.1e})"
    )

## Choosing the penalty

The penalty multiplies the squared slopes against the summed squared errors, so
the same $\alpha$ shrinks less as the training sample grows. It is chosen once
per asset, from a grid fixed in advance, on the calibration window: the
forecasts made before the evaluation starts. Each penalty is run walk-forward
over that window and scored with QLIKE exactly as the evaluation will score it;
the lowest score wins. The grid is not widened after seeing the results, which
would tune on the test data.

In [3]:
RIDGE_GRID = (0.1, 1.0, 10.0, 100.0)
PREPARED = {
    asset: build_forecast_data(daily_dataset(asset, LAYOUT), [LOG_RV_MEANS], value_col=LOG_RV)
    for asset in ASSETS
}
CALIBRATION = {asset: calibration(prepared) for asset, prepared in PREPARED.items()}
RIDGE = {asset: ridge_har(scores.idxmin()) for asset, scores in CALIBRATION.items()}
display(_display.design_table(DESIGN, ASSETS))
display(model_table([HAR, *RIDGE.values()]))
pd.DataFrame({pair(asset): scores for asset, scores in CALIBRATION.items()}).round(4)

,study design
assets,EUR/USD
minimum training sessions,252
first forecast target,2024-10-01
evaluation start,2025-01-01
last forecast target,2026-09-30
candidate model,HAR
loss,QLIKE


,rule,settings,regressors
model,,,
HAR,LinearRegression,ridge = 0.0,"log_rv_mean_1, log_rv_mean_5, log_rv_mean_22"
HAR-ridge,LinearRegression,ridge = 100.0,"log_rv_mean_1, log_rv_mean_5, log_rv_mean_22"


,EUR/USD
ridge penalty,
0.1,0.3073
1.0,0.3073
10.0,0.3066
100.0,0.3031


The choice must not depend on the evaluation period, or HAR-ridge's result
would be tuned on its own test data. The next cell checks it: calibrating again
on the data before the evaluation start alone must give exactly the same
scores, and so the same penalty. It stops the notebook if not.

In [4]:
for asset, prepared in PREPARED.items():
    known = prepared.loc[prepared["target_date"] < pd.Timestamp(DESIGN.evaluation_start)]
    pd.testing.assert_series_equal(calibration(known), CALIBRATION[asset])
    print(f"{pair(asset)}: the calibration uses no evaluation data")

EUR/USD: the calibration uses no evaluation data


In [5]:
RUNS = {asset: DESIGN.run(PREPARED[asset], [HAR, RIDGE[asset]], series=asset) for asset in ASSETS}

## Answer

In [6]:
for asset in ASSETS:
    print(verdict(asset))

EUR/USD: chosen penalty 100 (the edge of the grid); HAR-ridge loses to HAR: HAR's mean QLIKE is 6.7% below HAR-ridge's (DM p-value 7.1e-04)


If small penalties match HAR and the chosen one loses, the lesson is about
tuning, not about ridge: a few months of forecasts are too few to choose a
penalty, and a calibration score that keeps falling towards the edge of the
grid is a warning that the grid, or the window, cannot locate an optimum.

## 1. HAR-ridge against HAR

The study tests every model against HAR, so the DM p-value asks whether HAR is
better than HAR-ridge; a small value means the penalty hurt.

In [7]:
by_asset({asset: results_table(run.evaluation) for asset, run in RUNS.items()}).style.format(
    {LOSS_LABEL: "{:.4f}", LOWER: "{:.1%}", "DM p-value": "{:.1e}", WINS: "{:.0%}"}, na_rep="—"
)

,QLIKE,HAR lower by,DM p-value,HAR wins
model,,,,
HAR,0.1525,0.0%,—,—
HAR-ridge,0.1635,6.7%,7.1e-04,64%


## 2. With hindsight

Was the method wrong, or only the choice of penalty? Scoring every penalty in
the grid on the evaluation period answers that. This uses the evaluation data,
so it chooses nothing: it only shows what the calibration window could not.

In [8]:
HINDSIGHT = {asset: hindsight(run) for asset, run in RUNS.items()}
for asset, scores in HINDSIGHT.items():
    har = results_table(RUNS[asset].evaluation).loc[CANDIDATE, LOSS_LABEL]
    best = scores.idxmin()
    print(
        f"{pair(asset)}: {CANDIDATE} without a penalty {har:.4f}; best penalty in hindsight "
        f"{best:g} at {scores[best]:.4f}, {scores[best] / har - 1:+.3%} against {CANDIDATE}"
    )
pd.DataFrame({pair(asset): scores for asset, scores in HINDSIGHT.items()}).round(4)

EUR/USD: HAR without a penalty 0.1525; best penalty in hindsight 0.1 at 0.1525, +0.003% against HAR


,EUR/USD
ridge penalty,
0.1,0.1525
1.0,0.1525
10.0,0.1531
100.0,0.1635


**Caveats.** The penalty is in units of summed squared regressors, so its
strength drifts as the training sample grows; a penalty per observation would
keep it constant and is a natural next variant.